In [ ]:
%matplotlib ipympl
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import sys
import numpy as np
import os
import time
from IPython.display import clear_output, display, Image
sys.path.append('/Users/orenm/Desktop/code_projects/BlenderShaderProject/project_files/')

In [ ]:
from Logic.tree_networks_manager import TreesNetworkManager
from Logic.mcts_operator import MCTSOperator, search_metrics, get_nm_params_as_flat_array
from Logic.utils import show_image_grid

In [ ]:
path = '/Users/orenm/BlenderShaderProject/data/'
images_path = os.path.join(path, 'images/')
db_path = os.path.join(path, 'DB/')
temp_img_path = os.path.join(path, 'temp_images/')
active_models_path = os.path.join(path, "active_models/")
mcts_workdir = os.path.join(path, 'mcts_work_dir')
pintrest_images_dir = os.path.join(path, 'pintrest_images_processed_squares')

In [ ]:
db_manager = TreesNetworkManager.load(db_path, load_networks_managers=True)
len(db_manager.network)

In [ ]:
correction_model = 'balanced_models_ep_2_code_corrector_be_small_ru_3_we_0_001_le_5e-06.pt'
image_embedder_for_texture = 'ep_4_la_7_256_le_0_0001_mo_resnet_fi_128_sc_cosine.pt'
code_emb_file_name = 'ep_12_code_emb_be_mine_we_0_001_le_1e-05_be_big_3.pt'
image_emb_file_name = code_emb_file_name.replace('code_emb', 'image_emb')
correction_model_path = os.path.join(active_models_path, correction_model)
image_embedder_for_texture_path = os.path.join(active_models_path, image_embedder_for_texture)
tokenizer_path = os.path.join(active_models_path, "my_tokenizer")

mcts_operator = MCTSOperator(correction_model_path, tokenizer_path, image_embedder_for_texture_path)

In [ ]:
images_to_test_on = ['21633', '125953', '76803', '98059', '44811', '175119', '169872', '65551', '148626', '170387', '125715', '109588',
                     '101529', '49945', '124446', '154398', '150818', '53413', '18469', '15015', '64681', '145706', '116395', '162091',
                     '107565', '141998', '76079', '71216', '58033', '27700', '31481', '59318', '59702', '52793', '164922', '47803', '170938',
                     '15293', '59074', '143171', '126276', '46277', '39750', '170567', '167369', '19663', '51792', '16595', '64341', '172503',
                     '107608', '163033', '64602', '129242', '21723', '39901', '152413', '151265', '158183', '91115', '57196', '51565', '45295',
                     '115825', '154737', '46580', '34805', '155767', '49784', '159993', '49276', '151806', '57041', '43225', '159596', '12101',
                    '89387', '45401', '38743', '25682', '128718', '132416', '37178', '110873', '92773', '101139', '60374', '35174', '9395',
                    '144665', '9295', '45952']
len(images_to_test_on)

In [ ]:
non_empty = set(db_manager.get_nodes_without_label(IS_EMPTY_IMAGE))
have_network = set(db_manager.get_nodes_without_label(IS_EMPTY_IMAGE))
cluster_base = set(db_manager.get_nodes_without_label(IS_CLUSTER_BASE))
relevant_nodes = list(set.intersection(non_empty, have_network, cluster_base))
relevant_nodes = images_to_test_on

nodes_to_show = np.random.choice(relevant_nodes, min(16, len(relevant_nodes)), replace=False)
nodes_data = [(node_id, '') for node_id in nodes_to_show]
img_paths = [(os.path.join(images_path, f"{node_id}.png"), f'{text}, {node_id}') for node_id, text in nodes_data]
show_image_grid(img_paths)

In [ ]:
target_img_path = db_manager.make_image_path('45952', images_path)

In [ ]:
t = time.time()
graph_manager, all_node_expansions = mcts_operator.search(
    target_img_path, mcts_workdir,
    max_expansions=5,
    max_nodes_to_expand_per_iter=7,
    search_method="mcts",
    c_puct=3.3,
    # temperature = 0.1,
    sample_labels=True,
    optimize=True,
    n_nodes_to_optimize_at_end=3,
)
clear_output()
print(f'time: {time.time() - t}')

In [ ]:
# collect and show the best
best_nodes = graph_manager.get_nodes_sorted(TEXTURE_SIMILARITY_VALUE)
best_nodes = [(node_id, score) for score, node_id in best_nodes if score != float('-inf')
              and mcts_operator.graph_manager.node_has_label(node_id, HAS_IMAGE)]
images_data = []
for node_id, score in best_nodes:
    images_data.append((mcts_operator.graph_manager.make_image_path(node_id, mcts_operator.work_dir), f'node: {node_id}, score: {score:.2}'))
show_image_grid(images_data)

In [ ]:
display.Image(filename=target_img_path)